# Cropy 작업자 분리 시험셋 구축과 기준선 학습

공식 Validation은 Training과 같은 작업자·같은 날짜 사진이라(4,070장 중 4,063장) 검증 99.6%가 실제 성능을 부풀립니다.
이 노트북은 **촬영 작업자 단위로** 학습/검증/시험을 다시 나누고, 시험 작업자 사진을 한 장도 보지 않은 기준선 모델을 만듭니다.

- 분할: `MyDrive/cropy_testset/workers.json` (작업자 → train/val/test, Claude가 설계). 9_기타의 AI Hub 조각은 원본 사진의 작업자를 따르고, COCO 조각은 내용 해시로 70/10/20 나눕니다.
- 학습: 서버 모델과 같은 설정(ResNet50, 256px, 흐림 증강, 클래스 균형, ImageNet 초기화), 8 epoch. **T4 기준 1.5~2시간.** epoch마다 Drive에 저장되며 끊기면 위에서부터 다시 실행하면 이어집니다.
- 평가: 시험 작업자 사진으로 새 모델과 현재 서버 모델을 비교합니다(서버 모델은 이 작업자들을 학습했으므로 참고용).
- 결과: `MyDrive/cropy_testset/baseline_v1/` (`best_model.pth`, `test_report.json`, `predictions_*.csv`, `split_manifest.csv`)

**GPU 런타임** 에서 위에서부터 차례로 실행하세요.

In [ ]:
from pathlib import Path
import csv, hashlib, json, re, shutil, subprocess, sys, zipfile
import numpy as np, torch
from google.colab import drive

assert torch.cuda.is_available(), 'GPU 런타임을 선택한 뒤 다시 실행하세요.'
print('GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')
MY = Path('/content/drive/MyDrive')
TS = MY / 'cropy_testset'
RUN = TS / 'baseline_v1'
WORK = Path('/content/cropy_testset')
DATA = WORK / 'dataset'
SPLIT = WORK / 'split'
EPOCHS, BATCH_SIZE, LR = 8, 16, 1e-4
EXPECTED_BASE_SHA256 = '52dea0080285d76b6cbc09c2659c052a18f20ef620d05bca462203e64e6872ee'
EXPECTED_CODE_SHA256 = 'efe7034c6acd0544416e0cdb94c8fe748081921d73cf2fd1513fbc9b55e292a4'
SOURCES = {'base': MY / 'cropy_full_512/cropy_full_512.zip', 'code': MY / 'cropy_other10/training_code_other10.zip',
           'aihub': MY / 'cropy_other10/aihub_other_v1.zip', 'coco': MY / 'cropy_other10/coco_other_v1.zip',
           'server': MY / 'cropy_other10/cropy_other10_handoff_20261027/full512_resnet50_other10_v1.pth',
           'workers': TS / 'workers.json'}
for path in SOURCES.values():
    assert path.is_file(), f'Drive에 파일이 없습니다: {path}'
WORK.mkdir(parents=True, exist_ok=True)
RUN.mkdir(parents=True, exist_ok=True)

## 1. 데이터 준비와 작업자 단위 재분할
Drive에서 로컬로 복사해 풀고, 모든 사진을 작업자 배정표대로 `split/{train,val,test}/<클래스>/`에 옮깁니다. AI Hub 9클래스 사진 수가 설계와 정확히 같은지 확인합니다.

In [ ]:
EXPECT = {"train": {"0": 589, "1": 528, "2": 267, "3": 124, "4": 574, "5": 310, "6": 5932, "7": 6726, "8": 5724}, "val": {"0": 134, "1": 166, "2": 59, "3": 49, "4": 107, "5": 58, "6": 1440, "7": 2023, "8": 1688}, "test": {"0": 361, "1": 330, "2": 212, "3": 78, "4": 230, "5": 125, "6": 2313, "7": 3349, "8": 3087}}

def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def unpack(src, dest, expected=None):
    local = WORK / src.name
    if not local.exists():
        shutil.copyfile(src, local)
    if expected:
        assert digest(local) == expected, f'해시 불일치: {src.name}'
    with zipfile.ZipFile(local) as z:
        for item in z.infolist():
            assert (dest / item.filename).resolve().is_relative_to(dest.resolve()), '잘못된 압축 경로'
        z.extractall(dest)

unpack(SOURCES['code'], WORK / 'code', EXPECTED_CODE_SHA256)
ASSIGN = json.loads(SOURCES['workers'].read_text(encoding='utf-8'))['assign']
GROUP = re.compile(r'(?:^|_)([0-9A-Za-z]+)_(20\d{6})(?=_|\.|$)')
def split_of(path):
    m = GROUP.search(path.name)
    if m:
        return ASSIGN[m.group(1)] if m.group(1) in ASSIGN else 'train'
    if path.parent.name.startswith('9_'):  # COCO 조각: 내용 해시로 70/10/20
        b = int(digest(path)[:8], 16) % 10
        return 'train' if b < 7 else 'val' if b == 7 else 'test'
    return 'train'

if not (WORK / 'split_done').exists():
    shutil.rmtree(DATA, ignore_errors=True)
    shutil.rmtree(SPLIT, ignore_errors=True)
    unpack(SOURCES['base'], WORK, EXPECTED_BASE_SHA256)
    (DATA / 'manifest.csv').rename(DATA / 'manifest_9class.csv')
    unpack(SOURCES['aihub'], WORK)
    unpack(SOURCES['coco'], WORK)
    rows = []
    for src_split in ('train', 'val'):
        for path in sorted((DATA / 'images' / src_split).glob('*/*')):
            s = split_of(path)
            dest = SPLIT / s / path.parent.name / path.name
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(path), dest)
            m = GROUP.search(path.name)
            rows.append([s, path.parent.name, path.name, m.group(1) if m else '', src_split])
    with (RUN / 'split_manifest.csv').open('w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(['split', 'class', 'file', 'worker', 'official_split'])
        w.writerows(rows)
    (WORK / 'split_done').touch()

counts = {s: {c.name: len(list(c.iterdir())) for c in sorted((SPLIT / s).iterdir())} for s in ('train', 'val', 'test')}
for s, cs in counts.items():
    print(s, sum(cs.values()), cs)
for s, cs in EXPECT.items():
    for c, n in cs.items():
        got = counts[s][[k for k in counts[s] if k.startswith(c + '_')][0]]
        assert got == n, f'{s}/{c}: {got} != 설계 {n}'
print('AI Hub 9클래스 사진 수가 설계와 일치합니다.')
fingerprint = ''.join(f'{name}: {digest(WORK / SOURCES[name].name)}\n' for name in ('base', 'aihub', 'coco'))
fingerprint += f"workers: {digest(SOURCES['workers'])}\n"
(SPLIT.parent / 'manifest.csv').write_text(fingerprint, encoding='utf-8')  # 학습 스크립트는 data-dir 상위의 manifest.csv를 읽는다
from torchvision.datasets import ImageFolder
train_ds, val_ds = ImageFolder(SPLIT / 'train'), ImageFolder(SPLIT / 'val')
assert train_ds.classes == val_ds.classes and len(train_ds.classes) == 10, train_ds.classes
EXPECT_IMAGES = (len(train_ds), len(val_ds))
(RUN / 'split_counts.json').write_text(json.dumps(counts, ensure_ascii=False, indent=1), encoding='utf-8')
print('학습/검증:', EXPECT_IMAGES)

## 2. 기준선 학습 (ImageNet에서 시작)
서버 모델과 같은 설정입니다. 검증은 학습에 없는 작업자의 사진이라 점수가 이전보다 낮게 나오는 것이 정상입니다.

In [ ]:
command = [sys.executable, '-u', str(WORK / 'code/train_full_colab.py'),
    '--data-dir', str(SPLIT), '--output-dir', str(RUN / 'main'),
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--lr', str(LR),
    '--num-workers', '2', '--img-size', '256', '--arch', 'resnet50', '--blur-aug',
    '--expect-images', str(EXPECT_IMAGES[0]), str(EXPECT_IMAGES[1])]
if (RUN / 'main/last_checkpoint.pth').exists():
    command.append('--resume')
print('이어 학습:', '--resume' in command)
subprocess.run(command, check=True)
shutil.copyfile(RUN / 'main/best_model.pth', RUN / 'best_model.pth')

## 3. 시험 작업자 사진으로 평가
새 모델과 현재 서버 모델의 10클래스 확률을 저장하고 지표를 계산합니다. 서버 모델은 시험 작업자를 학습했으므로 그 수치는 참고용입니다.

In [ ]:
from PIL import Image
from torchvision import models, transforms

def load(path):
    ckpt = torch.load(path, map_location='cpu', weights_only=True)
    model = models.resnet50(weights=None)
    model.fc = torch.nn.Linear(model.fc.in_features, ckpt['num_classes'])
    model.load_state_dict(ckpt['model_state_dict'])
    return model.cuda().eval(), ckpt

def predict(model, size, files):
    tf = transforms.Compose([transforms.Resize((size, size)), transforms.ToTensor(),
                             transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    out = []
    for i in range(0, len(files), 64):
        batch = files[i:i + 64]
        with torch.no_grad():
            out.append(model(torch.stack([tf(Image.open(p).convert('RGB')) for p in batch]).cuda()).softmax(1).cpu().numpy())
    return np.concatenate(out)

def metrics(y, pred, workers, ids):
    cm = np.zeros((len(ids), len(ids)), dtype=np.int64)
    for a, b in zip(y, pred):
        cm[ids.index(a), ids.index(b)] += 1
    recall = cm.diagonal() / np.maximum(cm.sum(1), 1)
    precision = cm.diagonal() / np.maximum(cm.sum(0), 1)
    f1 = 2 * recall * precision / np.maximum(recall + precision, 1e-9)
    per_worker = {}
    for w in sorted(set(workers)):
        sel = np.array([x == w for x in workers])
        per_worker[w or 'coco'] = dict(n=int(sel.sum()), accuracy=round(float((pred[sel] == y[sel]).mean()), 4))
    return dict(n=int(len(y)), accuracy=round(float((pred == y).mean()), 4), macro_f1=round(float(f1.mean()), 4),
                recall={str(c): round(float(r), 4) for c, r in zip(ids, recall)}, confusion=cm.tolist(), per_worker=per_worker)

report = {}
for name, path in (('baseline', RUN / 'best_model.pth'), ('server', SOURCES['server'])):
    model, ckpt = load(path)
    ids = [int(c.split('_', 1)[0]) for c in ckpt['classes']]
    for split in ('test', 'val'):
        files = sorted((SPLIT / split).glob('*/*'))
        probs = predict(model, ckpt.get('img_size', 256), files)
        y = np.array([int(p.parent.name.split('_')[0]) for p in files])
        pred = np.array(ids)[probs.argmax(1)]
        workers = [(GROUP.search(p.name) or [None, ''])[1] for p in files]
        report[f'{name}_{split}'] = metrics(y, pred, workers, ids)
        with (RUN / f'predictions_{name}_{split}.csv').open('w', newline='', encoding='utf-8') as f:
            w = csv.writer(f)
            w.writerow(['file', 'label', 'worker'] + [f'p{i}' for i in ids])
            for p, wk, pr in zip(files, workers, probs):
                w.writerow([p.name, int(p.parent.name.split('_')[0]), wk] + [round(float(v), 5) for v in pr])
        r = report[f'{name}_{split}']
        print(f"{name} {split}: 정확도 {r['accuracy']:.2%}, macro F1 {r['macro_f1']:.3f}, 무노균병 recall {r['recall']['3']:.2%}")
    report[f'{name}_epoch'] = int(ckpt['epoch'])
    del model
(RUN / 'test_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=1), encoding='utf-8')
print('저장:', RUN / 'test_report.json')